# autoresearch-medimage - Colab CheXNet demo

Run the ChestX-ray14 `chexnet` baseline in Colab from a dataset copy you already have.

Recommended workflow:
- use Google Drive for the full dataset
- point `CHESTXRAY14_SOURCE` at a Drive folder or archive
- run `prepare.py`, `train.py`, and `finalize_run.py`

The repo setup path below assumes the repo is public and uses a public snapshot download. No GitHub token is required.

If you only want a fast runtime sanity check, use `notebooks/colab_quickstart.ipynb` first.

## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mlungrengmail/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies
!uv sync

## 2. Stage a local ChestX-ray14 copy

Use one of these inputs:
- a mounted Drive folder that already contains ChestX-ray14 files
- a mounted Drive archive such as `chestxray14.zip` or `chestxray14.tar.gz`
- an uploaded archive from your computer (works, but Drive is more reliable for the full dataset)

The staged data can be raw ChestX-ray14 files or a previously prepared `data/chestxray14` tree.

In [ ]:
# Optional: mount Google Drive for a full local dataset copy
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Optional: upload a local archive from your computer
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Set this to your local ChestX-ray14 source path
# Examples:
# CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14'
# CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14.zip'
# CHESTXRAY14_SOURCE = '/content/chestxray14.zip'
CHESTXRAY14_SOURCE = ''

In [ ]:
import shutil
import tarfile
import zipfile
from pathlib import Path

source = Path(CHESTXRAY14_SOURCE)
dest = Path('/content/autoresearch-medimage/data/chestxray14')

if not str(source).strip():
    raise ValueError('Set CHESTXRAY14_SOURCE to a Drive folder or archive before running this cell.')
if not source.exists():
    raise FileNotFoundError(f'ChestX-ray14 source not found: {source}')

if dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True, exist_ok=True)

if source.is_dir():
    shutil.copytree(source, dest, dirs_exist_ok=True)
elif source.suffix.lower() == '.zip':
    with zipfile.ZipFile(source) as archive:
        archive.extractall(dest)
elif source.name.endswith(('.tar.gz', '.tgz', '.tar')):
    with tarfile.open(source) as archive:
        archive.extractall(dest)
else:
    raise ValueError('Unsupported source type. Use a folder, .zip, .tar.gz, .tgz, or .tar archive.')

print(f'Staged ChestX-ray14 under: {dest}')
for candidate in ['Data_Entry_2017.csv', 'Data_Entry_2017_v2020.csv', 'train_val_list.txt', 'test_list.txt']:
    hits = list(dest.rglob(candidate))
    if hits:
        print(f'Found {candidate}: {hits[0]}')
image_count = sum(1 for p in dest.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
print(f'Image files found: {image_count}')

## 3. Prepare and run CheXNet

In [ ]:
!uv run prepare.py --dataset chestxray14

In [ ]:
# Run one 300-second baseline experiment
!uv run train.py --recipe chexnet --save-checkpoint

In [ ]:
# Finalize: held-out test eval, paper, figures, regulatory docs
!uv run python tools/finalize_run.py --dataset chestxray14

In [ ]:
# Inspect the latest ledger rows
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
df[df['dataset'] == 'chestxray14'][['timestamp', 'recipe', 'backbone', 'metric', 'status']].tail(10)